# Solution A1 · A classification tree from scratch: Gini, best split, recursion

**Type**: Implement from scratch

## Objective

Implement a classification tree with numpy: Gini impurity, search for the **best split** over all the features and thresholds (with cumulative sums, without recounting at every threshold), recursive construction with `max_depth` and `min_samples_leaf`, and prediction.
Check that the root, the structure and the predictions match `DecisionTreeClassifier`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=0)
sk = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)

## Commented solution

In [ ]:
def gini(y):
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - (p ** 2).sum()


def best_split(X, y, min_leaf):
    n, p = X.shape
    g0 = gini(y)
    best = (0, None, None)
    for j in range(p):
        o = np.argsort(X[:, j], kind="stable")
        xs, ys = X[o, j], y[o]
        cs = np.cumsum(np.eye(2)[ys], axis=0)                              # cumulative class counts: left of each position
        total = cs[-1]
        for i in range(min_leaf - 1, n - min_leaf):                         # children with at least min_leaf examples
            if xs[i] == xs[i + 1]:
                continue                                                    # no threshold between equal values
            left, right = cs[i], total - cs[i]
            nl, nr = left.sum(), right.sum()
            gain = g0 - (nl * (1 - ((left / nl) ** 2).sum()) + nr * (1 - ((right / nr) ** 2).sum())) / n
            if gain > best[0] + 1e-12:
                best = (gain, j, (xs[i] + xs[i + 1]) / 2)
    return best

In [ ]:
def build(X, y, max_depth, min_leaf, depth=0):
    counts = np.bincount(y, minlength=2)
    if depth == max_depth or counts.max() == len(y):
        return {"leaf": counts}
    _, j, s = best_split(X, y, min_leaf)
    if j is None:
        return {"leaf": counts}
    m = X[:, j] <= s
    return {"feature": j, "threshold": s,
            "left": build(X[m], y[m], max_depth, min_leaf, depth + 1),
            "right": build(X[~m], y[~m], max_depth, min_leaf, depth + 1)}


def predict(tree, X):
    def one(x):
        node = tree
        while "leaf" not in node:
            node = node["left"] if x[node["feature"]] <= node["threshold"] else node["right"]
        return node["leaf"].argmax()
    return np.array([one(x) for x in X])


def count_leaves(tree):
    return 1 if "leaf" in tree else count_leaves(tree["left"]) + count_leaves(tree["right"])

In [ ]:
tree = build(X_tr, y_tr, 4, 5)
train_agreement = float((predict(tree, X_tr) == sk.predict(X_tr)).mean())
test_agreement = float((predict(tree, X_te) == sk.predict(X_te)).mean())
same_root = bool(tree["feature"] == sk.tree_.feature[0] and abs(tree["threshold"] - sk.tree_.threshold[0]) < 1e-3)
same_leaves = bool(count_leaves(tree) == sk.get_n_leaves())
print(f"agreement with scikit-learn: training {train_agreement:.3f}, test {test_agreement:.3f} | same root: {same_root} ({data.feature_names[tree['feature']]} <= {tree['threshold']:.2f}) | leaves: {count_leaves(tree)} against {sk.get_n_leaves()}")
print(f"test accuracy of your tree: {(predict(tree, X_te) == y_te).mean():.3f}")

## Check

In [ ]:
with verify():
    check(close(gini(np.array([0, 0, 1, 1])), 0.5) and close(gini(np.array([1, 1, 1])), 0.0) and close(gini(np.array([0, 0, 0, 1])), 0.375), "Gini: 0.5 for half and half, 0 for a pure node, 0.375 for 3 against 1", "1 - sum(p ** 2)")
    g, f, s = best_split(X_tr, y_tr, 5)
    check(f == sk.tree_.feature[0] and abs(s - sk.tree_.threshold[0]) < 1e-3 and g > 0.3, "the best split at the root matches the one of scikit-learn", "cumulative sums on the sorted data; threshold = midpoint")
    check(best_split(np.array([[1.0], [1.0], [1.0], [1.0]]), np.array([0, 1, 0, 1]), 1) == (0, None, None), "no split possible with a constant feature", "consider only the points where the value changes")
    check(train_agreement == 1.0 and test_agreement >= 0.98, "identical predictions on the training set and at least 98% on the test", "scikit-learn compares in float32: a few thresholds may differ")
    check(same_root is True and same_leaves is True and count_leaves(tree) == sk.get_n_leaves(), "same root and same number of leaves", "count_leaves(tree) == sk.get_n_leaves()")
    small = build(X_tr, y_tr, 1, 5)
    check(count_leaves(small) == 2 and (predict(small, X_tr) == DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_tr, y_tr).predict(X_tr)).all(), "with max_depth=1 (a stump): two leaves and the same predictions as scikit-learn", "maximum depth 1")
done()

## What you learned

- **A tree is an exhaustive search over features and thresholds.** With cumulative sums on the sorted data the gain of each threshold is computed in constant time: the cost of a node is that of sorting every feature, $O(p\,n\log n)$. This is why trees are fast to train even with many rows.
- The thresholds are **midpoints** between consecutive values, and there is none between equal values: with discrete features (age, counts) there are few thresholds and the tree is less flexible.
- The comparison with scikit-learn is exact on the training set and almost exact on the test: scikit-learn works in **`float32`**, so a point very close to a threshold can end up on the other side. In a dataset with continuous values it rarely happens (here it does not), with discrete values more often: a detail to remember when comparing implementations.
- The recursion is **greedy**: each node maximizes the immediate gain. The resulting tree is not the best possible at that depth, but it is almost always good and costs very little.

## Common pitfall

Computing the gain of each threshold by recounting the classes with a loop over the examples: $O(n^2)$ per feature. The version with cumulative sums is linear after the sorting, and it is the idea that makes trees practical.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How does a decision tree choose the split and what is the cost?</summary>

For each feature it sorts the examples and tries all the thresholds between consecutive different values, computing the impurity (Gini or entropy) of the two children with cumulative counts; it picks the feature-threshold pair with the maximum gain and repeats recursively on the children until the stopping criterion.
The cost per node is dominated by the sorting, O(p n log n); overall it is O(p n log n) per level of the tree. It is a greedy algorithm: it does not guarantee the optimal tree.

</details>